# 3551. 数位和排序需要的最小交换次数

[原文](https://leetcode.cn/problems/minimum-swaps-to-sort-by-digit-sum/solution/python3-pai-xu-hou-shu-quan-bfs-bing-cha-j0ku/)

## BFS

In [ ]:
# 按目标位置构图，每个环贡献环长减一。

In [ ]:
from collections import deque
from typing import List


class Solution:
    def minSwaps(self, nums: List[int]) -> int:
        def calc_dig_sum(num: int) -> int:
            return sum(int(digit) for digit in str(num))

        ordered_nums = sorted(nums, key=lambda num: (calc_dig_sum(num), num))
        target_index = {num: i for i, num in enumerate(ordered_nums)}
        # graph[i] 存放位置 i 能直接走到的邻居，本题只有一个。
        graph = [[target_index[num]] for num in nums]

        n = len(nums)
        visited = [False] * n

        def bfs(start: int) -> int:
            queue = deque([start])
            visited[start] = True  # 入队时签到，避免重复入队。
            count = 0

            while queue:
                current = queue.popleft()
                count += 1

                for neighbor in graph[current]:
                    if visited[neighbor]:
                        continue
                    visited[neighbor] = True
                    queue.append(neighbor)

            return count

        ans = 0
        for start in range(n):
            if not visited[start]:
                cycle_size = bfs(start)
                ans += cycle_size - 1

        return ans


# 保留 BFS 类，方便与后面的并查集解法一起验证。
SolutionBFS = Solution

## 并查集模板

In [ ]:
class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            # 路径折半：让当前节点直接指向祖父节点。
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        root_a, root_b = self.find(a), self.find(b)
        if root_a == root_b:
            return False

        # 小集合挂到大集合的根节点下。
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a
        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        self.components -= 1
        return True


## 并查集

In [ ]:
# 合并当前位置与目标位置，答案为元素数减连通分量数。

In [ ]:
from typing import List


class Solution:
    def minSwaps(self, nums: List[int]) -> int:
        def digit_sum(num: int) -> int:
            total = 0
            while num:
                total += num % 10
                num //= 10
            return total

        ordered_nums = sorted(nums, key=lambda num: (digit_sum(num), num))
        target_index = {num: i for i, num in enumerate(ordered_nums)}

        uf = UnionFind(len(nums))
        for i, num in enumerate(nums):
            uf.union(i, target_index[num])

        return len(nums) - uf.components


# 在力扣提交时，需同时复制上面的 UnionFind 类。


SolutionDSU = Solution


## 原文样例

In [ ]:
cases = [
    ([37, 100], 1),
    ([22, 14, 33, 7], 0),
    ([18, 43, 34, 16], 2),
    ([1], 0),
    ([40, 4, 13], 2),  # 数位和相同，按数字本身排序。
    ([20, 3, 1], 2),   # 上面逐步解释的三位置环。
]

for name, solver_class in [
    ('Solution 1: BFS', SolutionBFS),
    ('Solution 2: Union-Find', SolutionDSU),
]:
    solver = solver_class()
    for nums, expected in cases:
        actual = solver.minSwaps(nums)
        assert actual == expected, (name, nums, actual, expected)
    print(f'{name}: {len(cases)} examples passed; [20, 3, 1] -> {solver.minSwaps([20, 3, 1])}')